# Slow SVReg vs SIGReg — Controlled A/B Experiment (Google Colab)

Reproducible A/B pipeline comparing **SIGReg** (baseline) against **Slow SVReg**
(*"The Content–Geometry Tradeoff in World Models: When JEPA Learns Latent Maps"*):
- **SVReg**: unilateral directional variance floor $\mathbb{E}_u[\max(0, 1 - \text{std}_t(z_t \cdot u))^2]$ (no stop-gradient, optional IS `weights` for `PER + H-net`).
- **Slowness penalty**: $\beta \, \mathbb{E}_t \|b_{t+1} - b_t\|^2 / (2\,\text{tr}\,\text{Var}(b))$ on the last $1/8$ of latent dimensions ($\beta = 1/\lambda$).
- **Gated Residual Predictor**: $\hat{z}_{t+1} = z_t + c \odot f(z_{\le t}, a_t)$ with learnable gate initialized at $c_0 = 0.1$ and identity `pred_proj: null`.
- **Non-BatchNorm projector**: `module.GroupNorm1d` (or `torch.nn.LayerNorm`).

Covers:
1. **Lightweight `stable-envs` online render** (no pixel HDF5 GB downloads, no MuJoCo): `pusht_online` (~159 MB state-only) and `tworoom_online` (state-only).
2. **Standard benchmarks**: `MultiScaleDial` (`data=msdial`) and `Cube` (`data=ogb`, optional `mujoco`).
3. **Architectures**: `lewm` / `dinov3_lora` (`ARPredictor`) and `convnext_ms` / `convnext_ms_scratch` (`PyramidJEPA` + `ViTSpatialPredictor`).
4. **Evaluation**: CEM planning (`eval_snv.py` for `pusht_online`/`tworoom_online`, `eval.py` for `msdial`/`ogb`, plus `swm_msdial/paired_stats.py`) and physical probes (`probe_train.py --source emb` for Cube; `swm_msdial/probe.py` for MultiScaleDial).


## 1. Colab Setup, Environment Variables & GPU Check


In [ ]:
#@title 1.1 Check GPU & Disk Space
!nvidia-smi
!df -h /content


In [ ]:
#@title 1.2 Optional: Mount Google Drive for Persistent Cache / Checkpoints
USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/mwm_slow_svreg_ab"  #@param {type:"string"}

import os
from pathlib import Path

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SWM_ROOT = DRIVE_ROOT
else:
    SWM_ROOT = "/content/swm_home"

Path(SWM_ROOT, "datasets").mkdir(parents=True, exist_ok=True)
Path(SWM_ROOT, "checkpoints").mkdir(parents=True, exist_ok=True)

os.environ["STABLEWM_HOME"] = SWM_ROOT
os.environ["LOCAL_DATASET_DIR"] = SWM_ROOT
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Optional: set local/Drive DINOv3 checkpoint paths if already cached
# os.environ["DINOV3_CKPT"] = f"{SWM_ROOT}/pretrained/dinov3_vits16_pretrain_lvd1689m-08c60483.pth"
# os.environ["DINOV3_CONVNEXT_CKPT"] = f"{SWM_ROOT}/pretrained/dinov3_convnext_tiny_pretrain_lvd1689m-21b726bb.pth"

print("STABLEWM_HOME     =", os.environ["STABLEWM_HOME"])
print("LOCAL_DATASET_DIR =", os.environ["LOCAL_DATASET_DIR"])
print("MUJOCO_GL         =", os.environ["MUJOCO_GL"])


In [ ]:
#@title 1.3 Clone Repository (`convnext-stable-env`) & Install Dependencies
REPO_URL = "https://github.com/CHSTR/Multiscale-World-Models.git"  #@param {type:"string"}
BRANCH = "convnext-stable-env"  #@param {type:"string"}
REPO_DIR = "/content/Multiscale-World-Models"  #@param {type:"string"}
INSTALL_STABLE_ENVS = True   #@param {type:"boolean"}  # Lightweight online render for PushT & TwoRoom
INSTALL_MUJOCO = False       #@param {type:"boolean"}  # Only needed for Cube (`ogb`) eval

import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)

pkgs = [
    "git+https://github.com/galilai-group/stable-worldmodel.git",
    "hdf5plugin", "h5py", "einops", "wandb", "huggingface_hub",
    "pyarrow-hotfix", "pandas", "scipy", "tabulate", "scikit-learn",
]
if INSTALL_STABLE_ENVS:
    pkgs.append("git+https://github.com/galilai-group/stable-envs.git#egg=stable-envs[pusht,navigation]")
if INSTALL_MUJOCO:
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    subprocess.run(["apt-get", "-qq", "install", "-y", "swig", "build-essential"], check=False)
    pkgs.extend(["mujoco", "ogbench"])

subprocess.run(["pip", "install", "-q", *pkgs], check=True)

import torch, stable_worldmodel as swm
print("PyTorch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
print("stable_worldmodel:", swm.__file__)
if INSTALL_STABLE_ENVS:
    import stable_envs as snv
    print("stable_envs:", snv.__file__, "(PushT, TwoRoom ready)")


## 2. Dataset Setup (`stable-envs` Online, `MultiScaleDial` & `Cube`)

- **`pusht_online` / `tworoom_online`** *(recommended lightweight path for Colab)*: stores only states + actions (~10–160 MB) and renders 224x224 frames on-the-fly in GPU via `stable-envs` (`online_render.py`).
- **MultiScaleDial (`msdial`)**: generated locally into `$LOCAL_DATASET_DIR/datasets/msdial` (~13 GB full, or smaller for fast runs).
- **Cube (`ogbench/cube_single_expert.h5`)**: ~95 GB uncompressed from `quentinll/lewm-cube` (requires `INSTALL_MUJOCO=True` and >= 110 GB free).


In [ ]:
#@title 2.1 Prepare Lightweight `stable-envs` State-Only Datasets (`tworoom_online` / `pusht_online`)
PREPARE_TWOROOM_STATE = True   #@param {type:"boolean"}
TWOROOM_EPISODES = 400         #@param {type:"integer"}
PREPARE_PUSHT_STATE = False    #@param {type:"boolean"}
PUSHT_SOURCE = "collect"       #@param ["collect", "strip_from_hf"]
PUSHT_COLLECT_EPS = 400        #@param {type:"integer"}

import os, subprocess
from pathlib import Path

ds_dir = Path(os.environ["LOCAL_DATASET_DIR"]) / "datasets"
ds_dir.mkdir(parents=True, exist_ok=True)

tworoom_h5 = ds_dir / "tworoom_state.h5"
if PREPARE_TWOROOM_STATE:
    if not tworoom_h5.exists():
        subprocess.run([
            "python", "scripts/make_states_only_h5.py", "collect",
            "--env", "tworoom", "--episodes", str(TWOROOM_EPISODES),
            "--steps", "100", "--out", str(tworoom_h5),
        ], check=True)
    print(f"TwoRoom state-only ready: {tworoom_h5} ({tworoom_h5.stat().st_size / 1e6:.1f} MB)")

pusht_h5 = ds_dir / "pusht_state.h5"
if PREPARE_PUSHT_STATE:
    if pusht_h5.exists():
        print(f"Found existing PushT state-only: {pusht_h5} ({pusht_h5.stat().st_size / 1e6:.1f} MB)")
    elif PUSHT_SOURCE == "collect":
        subprocess.run([
            "python", "scripts/make_states_only_h5.py", "collect",
            "--env", "pusht", "--episodes", str(PUSHT_COLLECT_EPS),
            "--steps", "100", "--out", str(pusht_h5),
        ], check=True)
        print(f"PushT state-only collected: {pusht_h5} ({pusht_h5.stat().st_size / 1e6:.1f} MB)")
    else:
        # Download expert PushT from HF, strip pixels (~159 MB result), and remove heavy pixel file
        full_pusht = ds_dir / "pusht_expert_train.h5"
        if not full_pusht.exists():
            subprocess.run(["hf", "download", "quentinll/lewm-pusht", "--repo-type", "dataset", "--local-dir", str(ds_dir)], check=True)
            for zst in sorted(ds_dir.glob("**/*.h5.zst")):
                subprocess.run(["zstd", "-d", "-f", str(zst), "-o", str(full_pusht)], check=True)
        subprocess.run(["python", "scripts/make_states_only_h5.py", "strip", str(full_pusht), str(pusht_h5)], check=True)
        print(f"PushT expert state-only ready: {pusht_h5} ({pusht_h5.stat().st_size / 1e6:.1f} MB)")


In [ ]:
#@title 2.2 Prepare MultiScaleDial Dataset (Train / Val / Paired Eval)
PREPARE_MSDIAL = False   #@param {type:"boolean"}
MSDIAL_TRAIN_EPS = 1800  #@param {type:"integer"}
MSDIAL_VAL_EPS = 200     #@param {type:"integer"}
MSDIAL_EVAL_EPS = 200    #@param {type:"integer"}
MSDIAL_STEPS = 100       #@param {type:"integer"}

import os
from pathlib import Path

if PREPARE_MSDIAL:
    msdial_dir = Path(os.environ["LOCAL_DATASET_DIR"]) / "datasets" / "msdial"
    train_h5 = msdial_dir / "msdial_train_k2_balanced.h5"
    eval_fine_h5 = msdial_dir / "msdial_eval_k2_balanced_fine.h5"

    if not train_h5.exists():
        !python swm_msdial/collect.py --levels 2 --style balanced \
            --episodes {MSDIAL_TRAIN_EPS} --val-episodes {MSDIAL_VAL_EPS} --steps {MSDIAL_STEPS} \
            --out-dir "{msdial_dir}"
    else:
        print("Found existing:", train_h5)

    if not eval_fine_h5.exists():
        !python swm_msdial/gen_eval_set.py --levels 2 --style balanced \
            --episodes {MSDIAL_EVAL_EPS} --out-dir "{msdial_dir}"
    else:
        print("Found existing:", eval_fine_h5)
else:
    print("Skipping MultiScaleDial generation (set PREPARE_MSDIAL=True to enable).")


In [ ]:
#@title 2.3 Prepare Cube Dataset (`ogbench/cube_single_expert.h5`)
DOWNLOAD_CUBE = False  #@param {type:"boolean"}

import os, shutil, subprocess
from pathlib import Path

cube_dir = Path(os.environ["LOCAL_DATASET_DIR"]) / "datasets" / "ogbench"
cube_h5 = cube_dir / "cube_single_expert.h5"

free_gb = shutil.disk_usage(os.environ["LOCAL_DATASET_DIR"]).free / (1024**3)
print(f"Free space at {os.environ['LOCAL_DATASET_DIR']}: {free_gb:.1f} GB")

if cube_h5.exists():
    print(f"Found Cube dataset: {cube_h5} ({cube_h5.stat().st_size / (1024**3):.1f} GB)")
elif DOWNLOAD_CUBE:
    assert free_gb >= 110, f"Cube requires ~110 GB free (have {free_gb:.1f} GB)"
    cube_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["hf", "download", "quentinll/lewm-cube", "--repo-type", "dataset", "--local-dir", str(cube_dir)], check=True)
    for zst in sorted(cube_dir.glob("**/*.tar.zst")):
        subprocess.run(["tar", "--use-compress-program=unzstd", "-xvf", str(zst), "-C", str(cube_dir)], check=True)
    assert cube_h5.exists(), f"Expected {cube_h5} after extraction!"
else:
    print("Set DOWNLOAD_CUBE=True (or place cube_single_expert.h5 in Drive) to run Cube arms.")


## 3. Controlled A/B Matrix (`SIGReg` vs `Slow SVReg`)

Each arm uses identical dataset split, seed, optimizer, and batch size, varying only the representation recipe:
- **Baseline (`sigreg`)**: `loss=sigreg`, `gated_residual=false`, default `pred_proj`, default projector norm.
- **Treatment (`slow_svreg`)**: `+experiment=slow_svreg` (`loss=slow_svreg`, `loss.svreg.weight=30.0`, `loss.slow.weight=null` $\Rightarrow \beta = 1/\lambda$, `model.predictor.gated_residual=true`, `model.predictor.gate_init=0.1`, `model.pred_proj=null`, `model.projector.norm_fn._target_=module.GroupNorm1d`).


In [ ]:
#@title 3.1 Configure & Dry-Run A/B Matrix
# Options: "tworoom_online", "pusht_online", "msdial", "ogb"
DATASETS = ["tworoom_online"]  #@param {type:"raw"}
# Optional override for dataset filename per dataset (e.g. pusht_state.h5 for pusht_online)
DATA_NAME_OVERRIDES = {"pusht_online": "pusht_state.h5", "tworoom_online": "tworoom_state.h5"}  #@param {type:"raw"}
MODELS = ["lewm", "convnext_ms"]  #@param {type:"raw"}  # options: "lewm", "dinov3_lora", "convnext_ms", "convnext_ms_scratch"
REPLAY = "disabled"  #@param ["disabled", "per_is"]
EPOCHS = 10          #@param {type:"integer"}
SEED = 0             #@param {type:"integer"}
LAM = 30.0           #@param {type:"number"}
SLOW_WEIGHT = "null" #@param {type:"string"}  # "null" sets beta = 1 / LAM
NORM = "GroupNorm1d" #@param ["GroupNorm1d", "LayerNorm"]
NUM_EVAL = 50        #@param {type:"integer"}
PROBE_FRAMES = 50000 #@param {type:"integer"}

def build_run_name(dataset, model, loss_mode, replay=REPLAY, seed=SEED):
    return f"ab_{dataset}_{model}_{loss_mode}_r{replay}_s{seed}"

def launcher_cmd(ds, mdl, loss_mode, dry_run=False):
    rname = build_run_name(ds, mdl, loss_mode)
    cmd = [
        "bash", "scripts/run_ab_slow_svreg.sh",
        "--dataset", ds, "--model", mdl, "--loss", loss_mode, "--replay", REPLAY,
        "--epochs", str(EPOCHS), "--seed", str(SEED), "--lam", str(LAM),
        "--slow-weight", str(SLOW_WEIGHT), "--norm", NORM,
        "--num-eval", str(NUM_EVAL), "--probe-frames", str(PROBE_FRAMES),
        "--run-name", rname,
    ]
    if ds in DATA_NAME_OVERRIDES and DATA_NAME_OVERRIDES[ds]:
        cmd.extend(["--data-name", DATA_NAME_OVERRIDES[ds]])
    if dry_run:
        cmd.append("--dry-run")
    return cmd

import subprocess
# Dry-run all selected arms first (prints exact commands without running)
for ds in DATASETS:
    for mdl in MODELS:
        for loss_mode in ("sigreg", "slow_svreg"):
            subprocess.run(launcher_cmd(ds, mdl, loss_mode, dry_run=True), check=True)


In [ ]:
#@title 3.2 Execute Controlled A/B Training + Eval + Probe
RUN_TRAINING = False  #@param {type:"boolean"}

import subprocess
if RUN_TRAINING:
    for ds in DATASETS:
        for mdl in MODELS:
            for loss_mode in ("sigreg", "slow_svreg"):
                rname = build_run_name(ds, mdl, loss_mode)
                print(f"=== Running {rname} ===")
                subprocess.run(launcher_cmd(ds, mdl, loss_mode, dry_run=False), check=True)
else:
    print("Set RUN_TRAINING=True to launch the A/B runs.")


## 4. Detailed Evaluation, Paired Statistics & Physical Probes

Inspect CEM success rates (`results.jsonl`), run paired McNemar + bootstrap CI comparisons across `tworoom_online`, `pusht_online`, and `msdial`, and summarize Cube probing (`block_pos`, `block_yaw`, `gripper`, `gripper_contact`).


In [ ]:
#@title 4.1 Paired Stats (`SIGReg` vs `Slow SVReg`) on `results.jsonl`
import os
from pathlib import Path

swm_root = Path(os.environ["STABLEWM_HOME"])

for ds in DATASETS:
    if ds not in ("msdial", "pusht_online", "tworoom_online"):
        continue
    metric_key = "success_dial" if ds == "msdial" else "successes"
    for mdl in MODELS:
        run_sig = build_run_name(ds, mdl, "sigreg")
        run_slw = build_run_name(ds, mdl, "slow_svreg")
        jsonl_a = swm_root / "checkpoints" / run_sig / "results.jsonl"
        jsonl_b = swm_root / "checkpoints" / run_slw / "results.jsonl"
        if jsonl_a.exists() and jsonl_b.exists():
            print(f"=== Paired stats ({ds}, {mdl}, key={metric_key}): SIGReg vs Slow SVReg ===")
            !python swm_msdial/paired_stats.py \
                --run-a "{jsonl_a}" --run-b "{jsonl_b}" \
                --key {metric_key}
        else:
            print(f"Skipping paired_stats for {ds}/{mdl} (missing results.jsonl)")


In [ ]:
#@title 4.2 Summarize Cube Probe Results (`block_pos`, `block_yaw`, `gripper`, `gripper_contact`)
import os
import pandas as pd
from pathlib import Path

probe_dir = Path(os.environ["STABLEWM_HOME"]) / "probe_outputs"
csvs = sorted(probe_dir.glob("*.csv")) if probe_dir.exists() else []
if csvs:
    df = pd.concat([pd.read_csv(p) for p in csvs], ignore_index=True)
    focus = ["block_pos", "block_yaw", "gripper", "gripper_contact"]
    sub = df[df["group"].isin(focus)].sort_values(["group", "checkpoint", "source", "probe"])
    display(sub[["checkpoint", "source", "probe", "group", "dim", "mse_z", "mse_raw", "pearson"]])
else:
    print("No Cube probe CSVs found yet in", probe_dir)
